# BP-IF Reproducibility

This notebook contains the BP-IF implementation and the experiments reported in the paper:

1. Main BP-IF evaluation
2. Controlled anomaly-ratio analysis
3. Ablation analysis
4. Friedman tests using the reported dataset-level results

Upload a CSV dataset containing numeric features and a binary label column. If the label column is not detected automatically, set `LABEL_COLUMN` to its exact name. Normal observations must be labelled `0` and anomalies `1`; categorical labels can be handled by setting `BENIGN_LABEL`.

The BP-IF configuration is fixed at 50 branches, 10 trees per branch, a bootstrap ratio of 0.7, projection dimension 32, and IF subsample size 256.


In [ ]:
# Imports and reproducibility
import os
import random
import time

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import friedmanchisquare
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.random_projection import GaussianRandomProjection


def set_all_seeds(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def ensure_binary_labels(y):
    y = np.asarray(y).reshape(-1)
    unique_labels = np.unique(y)
    if len(unique_labels) != 2:
        raise ValueError(
            f"y must contain exactly two unique labels. Found: {unique_labels}"
        )
    if set(unique_labels) != {0, 1}:
        mapping = {unique_labels[0]: 0, unique_labels[1]: 1}
        y = np.vectorize(mapping.get)(y).astype(int)
    return y


set_all_seeds(42)


In [ ]:
# BP-IF implementation
def bp_if_scores(
    X_train,
    X_test,
    seed,
    n_branches=50,
    bootstrap_ratio=0.7,
    projection_dim=32,
    trees_per_branch=10,
    max_samples=256,
):
    """Fit BP-IF and return mean test scores and timing measurements."""
    rng = np.random.RandomState(seed)
    n_train, n_features = X_train.shape
    bootstrap_size = max(10, int(bootstrap_ratio * n_train))

    branch_scores = []
    train_time = 0.0
    test_time = 0.0
    projection_time = 0.0

    for branch in range(int(n_branches)):
        indices = rng.randint(0, n_train, size=bootstrap_size)
        X_bootstrap = X_train[indices]
        branch_seed = seed + 1000 * branch

        start = time.perf_counter()
        projection = GaussianRandomProjection(
            n_components=min(int(projection_dim), n_features),
            random_state=branch_seed,
        )
        projection.fit(X_bootstrap)
        Z_bootstrap = projection.transform(X_bootstrap).astype(np.float32)
        Z_test = projection.transform(X_test).astype(np.float32)
        projection_time += time.perf_counter() - start

        start = time.perf_counter()
        model = IsolationForest(
            n_estimators=int(trees_per_branch),
            max_samples=int(max_samples),
            contamination="auto",
            random_state=branch_seed,
            n_jobs=-1,
        )
        model.fit(Z_bootstrap)
        train_time += time.perf_counter() - start

        start = time.perf_counter()
        branch_scores.append(-model.decision_function(Z_test))
        test_time += time.perf_counter() - start

    scores = np.vstack(branch_scores).mean(axis=0)
    return scores, train_time, test_time, projection_time


def evaluate_bpif_one_run(X, y, test_ratio=0.25, seed=42):
    set_all_seeds(seed)
    X = np.asarray(X)
    y = ensure_binary_labels(y)

    if X.ndim != 2:
        raise ValueError("X must be a two-dimensional array.")

    X_train_raw, X_test_raw, _, y_test = train_test_split(
        X,
        y,
        test_size=float(test_ratio),
        stratify=y,
        random_state=seed,
    )

    start = time.perf_counter()
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_test = scaler.transform(X_test_raw).astype(np.float32)
    preprocessing_time = time.perf_counter() - start

    scores, train_time, test_time, projection_time = bp_if_scores(
        X_train,
        X_test,
        seed=seed,
        n_branches=BPIF_BRANCHES,
        bootstrap_ratio=BPIF_BOOTSTRAP_RATIO,
        projection_dim=BPIF_PROJECTION_DIM,
        trees_per_branch=BPIF_TREES_PER_BRANCH,
        max_samples=BPIF_MAX_SAMPLES,
    )

    return {
        "model": "BP-IF",
        "roc_auc": float(roc_auc_score(y_test, scores)),
        "pr_auc": float(average_precision_score(y_test, scores)),
        "preprocessing_s": float(preprocessing_time),
        "projection_s": float(projection_time),
        "train_s": float(train_time),
        "test_s": float(test_time),
        "total_s": float(
            preprocessing_time + projection_time + train_time + test_time
        ),
        "seed": int(seed),
    }


def evaluate_bpif_many_runs(X, y, n_runs=10, test_ratio=0.25, seed_start=42):
    rows = []
    for run in range(int(n_runs)):
        row = evaluate_bpif_one_run(
            X,
            y,
            test_ratio=test_ratio,
            seed=int(seed_start + run),
        )
        row["run"] = run
        rows.append(row)

    all_runs = pd.DataFrame(rows)
    summary = (
        all_runs.groupby("model", as_index=False)
        .agg(
            roc_auc_mean=("roc_auc", "mean"),
            roc_auc_std=("roc_auc", "std"),
            pr_auc_mean=("pr_auc", "mean"),
            pr_auc_std=("pr_auc", "std"),
            total_s_mean=("total_s", "mean"),
            total_s_std=("total_s", "std"),
            preprocessing_s_mean=("preprocessing_s", "mean"),
            projection_s_mean=("projection_s", "mean"),
            train_s_mean=("train_s", "mean"),
            test_s_mean=("test_s", "mean"),
        )
        .reset_index(drop=True)
    )
    return summary, all_runs


In [ ]:
# Dataset input
from google.colab import files

uploaded = files.upload()
csv_path = next(iter(uploaded))
data = pd.read_csv(csv_path)

LABEL_COLUMN = None
BENIGN_LABEL = "BENIGN"

if LABEL_COLUMN is None:
    candidates = [
        "label",
        "Label",
        "class",
        "Class",
        "target",
        "Target",
        "diagnosis",
        "y",
        "Y",
        "outcome",
        "Outcome",
    ]
    detected = [column for column in candidates if column in data.columns]
    if not detected:
        raise ValueError(
            "The label column was not detected. Set LABEL_COLUMN to its column name."
        )
    LABEL_COLUMN = detected[0]
elif LABEL_COLUMN not in data.columns:
    raise ValueError(f"LABEL_COLUMN '{LABEL_COLUMN}' is not present in the dataset.")

labels = data[LABEL_COLUMN]
if pd.api.types.is_numeric_dtype(labels):
    y_true = labels.to_numpy()
    valid_labels = y_true[~pd.isna(y_true)]
    unique_labels = np.unique(valid_labels)
    if set(unique_labels.tolist()) != {0, 1}:
        if len(unique_labels) != 2:
            raise ValueError(f"The label column is not binary: {unique_labels}")
        label_mapping = {unique_labels[0]: 0, unique_labels[1]: 1}
        y_true = np.vectorize(label_mapping.get)(y_true).astype(int)
    else:
        y_true = y_true.astype(int)
else:
    y_true = (
        labels.astype(str).str.upper() != str(BENIGN_LABEL).upper()
    ).astype(int).to_numpy()

features = data.drop(columns=[LABEL_COLUMN]).select_dtypes(include=[np.number]).copy()
finite_rows = np.isfinite(features.to_numpy()).all(axis=1)
features = features.loc[finite_rows]
y_true = y_true[finite_rows]

if features.shape[0] < 50:
    raise ValueError(f"Too few valid rows remain: {features.shape[0]}")

X_raw = features.to_numpy(dtype=np.float32)

print("Dataset:", csv_path)
print("Feature matrix:", X_raw.shape)
print("Labels:", y_true.shape)
print("Anomaly ratio:", float(y_true.mean()))


In [ ]:
# Main BP-IF experiment
TEST_RATIO = 0.25
N_RUNS = 10
SEED_START = 42

BPIF_BRANCHES = 50
BPIF_BOOTSTRAP_RATIO = 0.7
BPIF_PROJECTION_DIM = 32
BPIF_TREES_PER_BRANCH = 10
BPIF_MAX_SAMPLES = 256

bpif_summary, bpif_all_runs = evaluate_bpif_many_runs(
    X_raw,
    y_true,
    n_runs=N_RUNS,
    test_ratio=TEST_RATIO,
    seed_start=SEED_START,
)

print("BP-IF summary")
display(bpif_summary)

print("BP-IF results by run")
display(bpif_all_runs.sort_values("run").reset_index(drop=True))


In [ ]:
# Controlled anomaly-ratio analysis
ANOMALY_RATIOS = [0.01, 0.03, 0.05]
RATIO_N_RUNS = 5
RATIO_SEED = 42


def make_fixed_anomaly_ratio(X, y, target_ratio=0.01, seed=42):
    """Create a subset with the requested anomaly ratio by downsampling."""
    X = np.asarray(X)
    y = ensure_binary_labels(y)
    ratio = float(target_ratio)

    if not 0 < ratio < 1:
        raise ValueError("target_ratio must be between 0 and 1.")

    rng = np.random.RandomState(seed)
    normal_indices = np.where(y == 0)[0]
    anomaly_indices = np.where(y == 1)[0]

    available_normals = len(normal_indices)
    available_anomalies = len(anomaly_indices)
    normals_from_all_anomalies = int(
        np.floor(available_anomalies * (1.0 - ratio) / ratio)
    )

    if normals_from_all_anomalies <= available_normals:
        n_anomalies = available_anomalies
        n_normals = normals_from_all_anomalies
    else:
        n_normals = available_normals
        n_anomalies = int(np.floor(available_normals * ratio / (1.0 - ratio)))

    if n_normals < 1 or n_anomalies < 2:
        raise ValueError(
            f"The {ratio:.2%} ratio cannot be formed from the available classes."
        )

    selected_normals = rng.choice(normal_indices, size=n_normals, replace=False)
    selected_anomalies = rng.choice(anomaly_indices, size=n_anomalies, replace=False)
    selected = np.concatenate([selected_normals, selected_anomalies])
    rng.shuffle(selected)

    X_subset = X[selected]
    y_subset = y[selected]
    details = {
        "target_ratio": ratio,
        "achieved_ratio": float(y_subset.mean()),
        "n_samples": int(len(selected)),
        "n_normals": int((y_subset == 0).sum()),
        "n_anomalies": int((y_subset == 1).sum()),
    }
    return X_subset, y_subset, details


ratio_summaries = []
ratio_runs = []

for anomaly_ratio in ANOMALY_RATIOS:
    X_ratio, y_ratio, ratio_details = make_fixed_anomaly_ratio(
        X_raw,
        y_true,
        target_ratio=anomaly_ratio,
        seed=RATIO_SEED,
    )

    summary, runs = evaluate_bpif_many_runs(
        X_ratio,
        y_ratio,
        n_runs=RATIO_N_RUNS,
        test_ratio=TEST_RATIO,
        seed_start=RATIO_SEED,
    )

    for frame in (summary, runs):
        for key, value in ratio_details.items():
            frame[key] = value

    ratio_summaries.append(summary)
    ratio_runs.append(runs)

bpif_ratio_summary = pd.concat(ratio_summaries, ignore_index=True)
bpif_ratio_all_runs = pd.concat(ratio_runs, ignore_index=True)

print("BP-IF anomaly-ratio summary")
display(
    bpif_ratio_summary[
        [
            "target_ratio",
            "achieved_ratio",
            "n_samples",
            "n_normals",
            "n_anomalies",
            "roc_auc_mean",
            "roc_auc_std",
            "pr_auc_mean",
            "pr_auc_std",
            "total_s_mean",
        ]
    ].sort_values("target_ratio")
)


In [ ]:
# Ablation analysis
def bootstrap_only_scores(
    X_train,
    X_test,
    seed,
    n_branches=50,
    bootstrap_ratio=0.7,
    trees_per_branch=10,
    max_samples=256,
):
    rng = np.random.RandomState(seed)
    n_train = X_train.shape[0]
    bootstrap_size = max(10, int(bootstrap_ratio * n_train))
    branch_scores = []
    train_time = 0.0
    test_time = 0.0

    for branch in range(int(n_branches)):
        indices = rng.randint(0, n_train, size=bootstrap_size)
        X_bootstrap = X_train[indices]

        start = time.perf_counter()
        model = IsolationForest(
            n_estimators=int(trees_per_branch),
            max_samples=int(max_samples),
            contamination="auto",
            random_state=seed + 1000 * branch,
            n_jobs=-1,
        )
        model.fit(X_bootstrap)
        train_time += time.perf_counter() - start

        start = time.perf_counter()
        branch_scores.append(-model.decision_function(X_test))
        test_time += time.perf_counter() - start

    scores = np.vstack(branch_scores).mean(axis=0)
    return scores, train_time, test_time, 0.0


def projection_only_scores(
    X_train,
    X_test,
    seed,
    projection_dim=32,
    trees=10,
    max_samples=256,
):
    n_features = X_train.shape[1]

    start = time.perf_counter()
    projection = GaussianRandomProjection(
        n_components=min(int(projection_dim), n_features),
        random_state=seed,
    )
    projection.fit(X_train)
    Z_train = projection.transform(X_train).astype(np.float32)
    Z_test = projection.transform(X_test).astype(np.float32)
    projection_time = time.perf_counter() - start

    start = time.perf_counter()
    model = IsolationForest(
        n_estimators=int(trees),
        max_samples=int(max_samples),
        contamination="auto",
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(Z_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    scores = -model.decision_function(Z_test)
    test_time = time.perf_counter() - start
    return scores, train_time, test_time, projection_time


def evaluate_ablation_one_run(X, y, seed):
    set_all_seeds(seed)
    X = np.asarray(X)
    y = ensure_binary_labels(y)

    X_train_raw, X_test_raw, _, y_test = train_test_split(
        X,
        y,
        test_size=TEST_RATIO,
        stratify=y,
        random_state=seed,
    )

    start = time.perf_counter()
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_test = scaler.transform(X_test_raw).astype(np.float32)
    preprocessing_time = time.perf_counter() - start

    configurations = [
        (
            "Full BP-IF",
            bp_if_scores,
            {
                "n_branches": BPIF_BRANCHES,
                "bootstrap_ratio": BPIF_BOOTSTRAP_RATIO,
                "projection_dim": BPIF_PROJECTION_DIM,
                "trees_per_branch": BPIF_TREES_PER_BRANCH,
                "max_samples": BPIF_MAX_SAMPLES,
            },
        ),
        (
            "Bootstrap only",
            bootstrap_only_scores,
            {
                "n_branches": BPIF_BRANCHES,
                "bootstrap_ratio": BPIF_BOOTSTRAP_RATIO,
                "trees_per_branch": BPIF_TREES_PER_BRANCH,
                "max_samples": BPIF_MAX_SAMPLES,
            },
        ),
        (
            "Projection only",
            projection_only_scores,
            {
                "projection_dim": BPIF_PROJECTION_DIM,
                "trees": BPIF_TREES_PER_BRANCH,
                "max_samples": BPIF_MAX_SAMPLES,
            },
        ),
    ]

    rows = []
    for name, scorer, parameters in configurations:
        scores, train_time, test_time, projection_time = scorer(
            X_train,
            X_test,
            seed=seed,
            **parameters,
        )
        rows.append(
            {
                "configuration": name,
                "roc_auc": float(roc_auc_score(y_test, scores)),
                "pr_auc": float(average_precision_score(y_test, scores)),
                "preprocessing_s": float(preprocessing_time),
                "projection_s": float(projection_time),
                "train_s": float(train_time),
                "test_s": float(test_time),
                "total_s": float(
                    preprocessing_time + projection_time + train_time + test_time
                ),
                "seed": int(seed),
            }
        )
    return pd.DataFrame(rows)


ablation_frames = []
for run in range(N_RUNS):
    run_results = evaluate_ablation_one_run(X_raw, y_true, SEED_START + run)
    run_results["run"] = run
    ablation_frames.append(run_results)

ablation_all_runs = pd.concat(ablation_frames, ignore_index=True)
ablation_summary = (
    ablation_all_runs.groupby("configuration", as_index=False)
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        total_s_mean=("total_s", "mean"),
        total_s_std=("total_s", "std"),
    )
    .sort_values(["roc_auc_mean", "pr_auc_mean"], ascending=False)
    .reset_index(drop=True)
)

print("Ablation summary")
display(ablation_summary)

print("Ablation results by run")
display(ablation_all_runs.sort_values(["configuration", "run"]).reset_index(drop=True))


In [ ]:
# Statistical analysis of reported dataset-level results
reported_results = [
    ("Backdoor", "BP-IF", 0.888905, 0.526702),
    ("Backdoor", "EIF", 0.890163, 0.526649),
    ("Backdoor", "IF", 0.738922, 0.048341),
    ("Backdoor", "DIF", 0.902974, 0.452932),
    ("Cardio", "BP-IF", 0.944362, 0.585822),
    ("Cardio", "EIF", 0.943495, 0.585186),
    ("Cardio", "IF", 0.931247, 0.586967),
    ("Cardio", "DIF", 0.626411, 0.213539),
    ("Census", "BP-IF", 0.663422, 0.086472),
    ("Census", "EIF", 0.662914, 0.086397),
    ("Census", "IF", 0.597406, 0.070837),
    ("Census", "DIF", 0.632320, 0.084181),
    ("Cover", "BP-IF", 0.942796, 0.088644),
    ("Cover", "EIF", 0.940441, 0.085905),
    ("Cover", "IF", 0.874885, 0.058096),
    ("Cover", "DIF", 0.717659, 0.020077),
    ("MNIST", "BP-IF", 0.855618, 0.399295),
    ("MNIST", "EIF", 0.848118, 0.393916),
    ("MNIST", "IF", 0.805529, 0.280520),
    ("MNIST", "DIF", 0.634074, 0.173717),
]

statistics_data = pd.DataFrame(
    reported_results,
    columns=["dataset", "model", "roc_auc_mean", "pr_auc_mean"],
)


def friedman_test(data, metric, metric_name):
    wide = data.pivot(index="dataset", columns="model", values=metric).sort_index()
    wide = wide[["BP-IF", "IF", "EIF", "DIF"]]
    statistic, p_value = friedmanchisquare(
        wide["BP-IF"].to_numpy(),
        wide["IF"].to_numpy(),
        wide["EIF"].to_numpy(),
        wide["DIF"].to_numpy(),
    )
    return pd.DataFrame(
        [
            {
                "metric": metric_name,
                "friedman_statistic": float(statistic),
                "p_value": float(p_value),
                "significant_0.05": bool(p_value < 0.05),
            }
        ]
    )


friedman_roc = friedman_test(statistics_data, "roc_auc_mean", "AUC-ROC")
friedman_pr = friedman_test(statistics_data, "pr_auc_mean", "AUC-PR")
friedman_results = pd.concat([friedman_roc, friedman_pr], ignore_index=True)

print("Friedman omnibus tests")
display(friedman_results)
